# ClinIQ: Gemma 4 E2B Fine-tuning with Unsloth

**Hackathon:** Gemma 4 Good (Unsloth Track)\
**Task:** Clinical entity extraction from eICR summaries\
**Target:** Edge deployment on Jetson Orin Nano (8GB)

## 1. Install Dependencies

In [ ]:
import subprocess, os

# GPU check — fail fast on P100
gpu_name = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
print(f"GPU: {gpu_name}")
assert "P100" not in gpu_name, "P100 not compatible with Unsloth. Change to T4 x2 in Settings → Save & Run All."
print("GPU OK!")

# Disable HF telemetry that can timeout
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xf = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=1.5.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xf} peft trl triton unsloth
!pip install transformers==5.5.0
!pip install torchcodec 2>/dev/null || true
!pip install --no-deps --upgrade timm
import torch; torch._dynamo.config.recompile_limit = 64

## 2. Load Model

In [ ]:
from unsloth import FastModel
import torch, os

# Disable Unsloth's HF stats check that can timeout
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

# Use Kaggle-cached model (instant, no HuggingFace download)
e2b_path = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1"
e4b_path = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1"

if os.path.exists(e2b_path + "/config.json"):
    model_path = e2b_path
    print(f"Loading E2B from Kaggle cache: {model_path}")
elif os.path.exists(e4b_path + "/config.json"):
    model_path = e4b_path
    print(f"Loading E4B from Kaggle cache: {model_path}")
else:
    model_path = "unsloth/gemma-4-E2B-it"
    print(f"Downloading from HuggingFace: {model_path}")

model, tokenizer = FastModel.from_pretrained(
    model_name = model_path,
    dtype = None,
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False,
)

## 3. Configure LoRA

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

## 4. Load Training Data

In [ ]:
from datasets import load_dataset
import glob, os

# Auto-detect dataset path
jsonl_files = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
if jsonl_files:
    DATASET_PATH = os.path.dirname(jsonl_files[0])
    print(f"Dataset: {DATASET_PATH}")
else:
    # Fallback: list all inputs for debugging
    print("Available inputs:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files[:20]: print(f"  {os.path.join(root, f)}")
    raise FileNotFoundError("train.jsonl not found — add eicr-fhir-training-data dataset")

dataset = load_dataset("json", data_files={
    "train": f"{DATASET_PATH}/train.jsonl",
    "validation": f"{DATASET_PATH}/val.jsonl",
})
print(f"Train: {len(dataset['train'])}  Val: {len(dataset['validation'])}")

## 5. Format for Chat Template

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

# Our data is already in correct conversation format — skip standardize_data_formats

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print("Sample:", dataset["train"][0]["text"][:200])

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = None,  # Skip eval for speed
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,  # Quick proof — increase later
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

In [ ]:
# Show memory before training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# Show training stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
print(f"{trainer_stats.metrics['train_runtime']:.0f}s training time")
print(f"Peak memory = {used_memory} GB (training used {used_memory_for_lora} GB)")
print(f"Loss: {trainer_stats.training_loss:.4f}")
print("Quick proof complete — increase max_steps for full training")

## 7. Test Inference

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

test_input = """Patient: Maria Garcia
Gender: F
DOB: 1985-06-14
Location: Denver, CO 80202
Facility: Denver Health Medical Center (NPI: 1234567800)
Encounter: 2026-03-15
Reason: fever (39.2C), dry cough for 5 days, shortness of breath
Dx: COVID-19 (SNOMED 840539006)
Lab: SARS-CoV-2 RNA NAA+probe Ql (Resp) (LOINC 94500-6) - Detected
Vitals: Temp 39.2C, HR 92, RR 20, SpO2 95%, BP 128
Meds: nirmatrelvir 150 MG / ritonavir 100 MG (RxNorm 2599543)"""

# Gemma 4 chat template expects content as list of typed dicts
messages = [
    {"role": "user", "content": [
        {"type": "text", "text": "Extract clinical entities from this eICR summary. Output JSON with: patient demographics, conditions (SNOMED/ICD-10), labs (LOINC), medications (RxNorm), vitals, and a case summary. Include confidence scores. Output valid JSON only.\n\n" + test_input}
    ]},
]

inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", tokenize=True, return_dict=True,
).to("cuda")

outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True, temperature=1.0, top_p=0.95, top_k=64)
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("=== Model output ===")
print(response[:500])

import json
try:
    print("\nValid JSON!")
    print(json.dumps(json.loads(response), indent=2))
except:
    print("\nNot valid JSON - expected with only 60 training steps")

## 8. Save & Export

In [ ]:
# Save LoRA adapter (~50MB)
model.save_pretrained("cliniq_lora")
tokenizer.save_pretrained("cliniq_lora")

import os
lora_files = os.listdir("cliniq_lora")
total = sum(os.path.getsize(os.path.join("cliniq_lora", f)) for f in lora_files)
print(f"LoRA saved to cliniq_lora/ — {total/1024/1024:.1f} MB ({len(lora_files)} files)")
print("Download from Output tab, then convert to GGUF locally")